In [ ]:
import pandas as pd
import sys
sys.path.append('../')
from src.processing import ip_to_int, merge_ip_range

# Load Raw Data
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_df = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

# 1. Correct Data Types
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

# 2. Check and Drop Duplicates
initial_shape = fraud_df.shape[0]
fraud_df = fraud_df.drop_duplicates()
print(f"Removed {initial_shape - fraud_df.shape[0]} duplicate rows.")

# 3. Geolocation Mapping
fraud_df['ip_address_int'] = fraud_df['ip_address'].apply(ip_to_int)
# Drop rows where IP address conversion failed
fraud_df = fraud_df.dropna(subset=['ip_address_int'])
fraud_df['ip_address_int'] = fraud_df['ip_address_int'].astype(int)

# Merge
df_geo = merge_ip_range(fraud_df, ip_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Quantify Imbalance
class_counts = df_geo['class'].value_counts()
class_pct = df_geo['class'].value_counts(normalize=True) * 100
print(f"Class Imbalance:\n0 (Legit): {class_counts[0]} ({class_pct[0]:.2f}%)\n1 (Fraud): {class_counts[1]} ({class_pct[1]:.2f}%)")

# Plot Class Distribution
plt.figure(figsize=(5, 4))
sns.countplot(x='class', data=df_geo, palette='viridis')
plt.title('Fraud vs Non-Fraud Class Distribution')
plt.show()

# Analyze Fraud Patterns by Country (Top 10 Countries with highest fraud count)
fraud_by_country = df_geo[df_geo['class'] == 1]['country'].value_counts().head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=fraud_by_country.values, y=fraud_by_country.index, palette='magma')
plt.title('Top 10 Countries by Total Fraud Transactions')
plt.xlabel('Number of Fraudulent Incidents')
plt.show()